In [3]:
from efficientnet_pytorch import EfficientNet
import torch

# Load the EfficientNet model with the custom number of output classes
model_name = 'efficientnet-b0'  # Adjust the model variant if needed
model = EfficientNet.from_name(model_name, num_classes=29)

# Specify the path to your model's state dict
model_path = r"E:\gp_final\image\recogntion\EfficientNet\ASL_HandSignLang_EfficientNetB0_2.pth"

# Load the state dict
state_dict = torch.load(model_path, map_location=torch.device('cpu'))

# Remove the 'module.' prefix if it exists
new_state_dict = {}
for k, v in state_dict.items():
    if k.startswith('module.'):
        new_state_dict[k[7:]] = v
    else:
        new_state_dict[k] = v

# Load the cleaned state dict into the model
model.load_state_dict(new_state_dict)

# Now the model should be loaded correctly
print("Model loaded successfully!")


Model loaded successfully!


In [10]:
import cv2
import torch
from torchvision import transforms
from PIL import Image
import mediapipe as mp
import time

model.eval()  # Set the model to evaluation mode

# Setup MediaPipe Hands
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(static_image_mode=False,
                       max_num_hands=1,
                       min_detection_confidence=0.5,
                       min_tracking_confidence=0.5)

# Define the transformation
transform = transforms.Compose([
    transforms.Resize((128, 128)),  # Resize the image to 128x128
    transforms.ToTensor()           # Convert the image to a PyTorch tensor
])

# Initialize the webcam
cap = cv2.VideoCapture(0)
if not cap.isOpened():
    print("Error: Could not open webcam.")
    exit()

# Define label mapping
label_map = {
    0: 'A', 1: 'B', 2: 'C', 3: 'D', 4: 'E', 5: 'F', 6: 'G', 7: 'H', 8: 'I', 9: 'J',
    10: 'K', 11: 'L', 12: 'M', 13: 'N', 14: 'O', 15: 'P', 16: 'Q', 17: 'R', 18: 'S', 19: 'T',
    20: 'U', 21: 'V', 22: 'W', 23: 'X', 24: 'Y', 25: 'Z', 26: 'del', 27: 'nothing', 28: 'space'
}

# Variables to track the prediction, timing, and word construction
last_predicted_label = None
label_change_time = time.time()
consistency_duration = 3  # Duration for which the prediction must remain the same
word_buffer = []
last_hand_detected_time = time.time()
hand_detected = False

try:
    while True:
        ret, frame = cap.read()
        if not ret:
            print("Failed to capture frame. Exiting...")
            break

        # Convert the frame to RGB and then to a PIL Image
        image = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        
        # Apply the transformations
        input_tensor = transform(image)
        input_batch = input_tensor.unsqueeze(0)  # Create a mini-batch as expected by the model

        # Perform inference
        with torch.no_grad():
            output = model(input_batch)
            predicted_index = output.argmax(1).item()  # Get the index of the max log-probability
            predicted_label = label_map.get(predicted_index, '?')  # Map index to label

        # Check if the predicted label has changed
        if predicted_label != last_predicted_label:
            last_predicted_label = predicted_label
            label_change_time = time.time()
        elif time.time() - label_change_time >= consistency_duration:
            print(f"Confirmed Prediction: {predicted_label}")
            if predicted_label != 'nothing':
                word_buffer.append(predicted_label)
            last_predicted_label = None  # Reset after printing to prevent continuous printing

        # Use MediaPipe to draw hand landmarks on the frame
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = hands.process(frame_rgb)
        if results.multi_hand_landmarks:
            hand_detected = True
            last_hand_detected_time = time.time()
            for hand_landmarks in results.multi_hand_landmarks:
                mp.solutions.drawing_utils.draw_landmarks(frame, hand_landmarks, mp_hands.HAND_CONNECTIONS)
        else:
            hand_detected = False

        # Check if no hand detected for 5 seconds to end the word
        if not hand_detected and time.time() - last_hand_detected_time >= 5:
            if word_buffer:
                print(f"Word: {''.join(word_buffer)}")
                word_buffer = []

        cv2.putText(frame, f'Predicted Label: {predicted_label}', (10, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2, cv2.LINE_AA)
        cv2.putText(frame, f'Current Word: {"".join(word_buffer)}', (10, 100), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2, cv2.LINE_AA)
        cv2.imshow('Real-Time Sign Language Recognition', frame)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
finally:
    cap.release()
    cv2.destroyAllWindows()
    hands.close()


Confirmed Prediction: A
Confirmed Prediction: C
Confirmed Prediction: B
Confirmed Prediction: D
Confirmed Prediction: J
Confirmed Prediction: F
Word: ACBDJF
